# 02 — Memori, State, dan Model Context Protocol (MCP)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/11-ai-agent/02_memori_state_dan_mcp.ipynb)

Agen di notebook 01 "lupa" segalanya setelah tugas selesai, dan tool-nya hanya bisa dipakai dari notebook itu. Notebook ini membahas tiga hal yang membuat agen berguna dalam jangka panjang:

1. **Memori jangka pendek** (konteks satu sesi): riwayat terus membesar. Bagaimana mengelolanya?
2. **Memori jangka panjang** (antarsesi): agen menyimpan dan membaca catatan, misalnya preferensi pengguna.
3. **State tugas**: agen menulis dan memperbarui **rencana** (daftar langkah) untuk tugas panjang.
4. **Model Context Protocol (MCP)**: standar terbuka agar tool yang sama bisa dipakai oleh banyak aplikasi AI.

> Biaya API notebook ini diperkirakan kurang dari US$1.

## 0. Instalasi, API Key, dan Database

Seperti modul 08: simpan `ANTHROPIC_API_KEY` di **Colab Secrets** (ikon 🔑, aktifkan *Notebook access*) atau di file `.env` (salin dari `.env.example` di root repo). **Jangan menulis API key di kode.** Notebook ini tidak butuh GPU.

In [ ]:
%pip install -q "anthropic[mcp]>=1.9" python-dotenv pandas matplotlib

In [ ]:
import json
import os
import sqlite3
import urllib.request
from pathlib import Path

import anthropic
import pandas as pd
from dotenv import find_dotenv, load_dotenv

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"

MODEL = "claude-opus-5-5"
# Refusal fallback (beta): jika permintaan ditolak, server mencoba ulang di model cadangan (lihat modul 08)
FALLBACK = {"betas": ["server-side-fallback-2026-07-01"], "fallbacks": "default"}


def ambil_api_key():
    """Cari API key: environment variable / file .env (lokal), lalu Colab Secrets."""
    load_dotenv(find_dotenv(usecwd=True))
    if os.getenv("ANTHROPIC_API_KEY"):
        return os.getenv("ANTHROPIC_API_KEY")
    try:
        from google.colab import userdata
        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:  # bukan di Colab, atau Secret belum dibuat / belum diberi akses
        return None


api_key = ambil_api_key()
if not api_key:
    raise RuntimeError(
        "ANTHROPIC_API_KEY belum diatur.\n"
        "- Colab: ikon 🔑 Secrets → tambah ANTHROPIC_API_KEY → aktifkan Notebook access, lalu jalankan ulang sel ini.\n"
        "- Lokal: salin .env.example menjadi .env di root repo, lalu isi ANTHROPIC_API_KEY.")
client = anthropic.Anthropic(api_key=api_key)
print(f"✅ API key ditemukan | model: {MODEL}")

In [ ]:
# File pendukung modul ini (ada di repo; di Colab diunduh dari GitHub)
REPO_RAW = "https://raw.githubusercontent.com/riki-profile/ai-engineer-journey/main/11-ai-agent"
for nama_file in ["toko_db.py", "server_toko.py"]:
    if not Path(nama_file).exists():
        urllib.request.urlretrieve(f"{REPO_RAW}/{nama_file}", nama_file)

from toko_db import buat_database  # noqa: E402  (diimpor setelah file tersedia)

DB = buat_database("toko.db")
print(f"Database: {DB} ({DB.stat().st_size / 1e6:.1f} MB)")

### Komponen dari notebook 01

Helper, tool SQL baca-saja, dan loop agen generik `jalankan_agen(pesan, tools, fungsi, system, ...)`. Isinya sama dengan notebook 01, ditambah dua hal: bisa melanjutkan riwayat `messages` yang sudah ada, dan punya batas **anggaran** (`max_biaya`).

In [ ]:
HARGA_PER_JUTA = {"input": 4.00, "output": 20.00}   # USD, claude-opus-5-5 (cek halaman harga resmi)


def biaya(usage):
    return (usage.input_tokens * HARGA_PER_JUTA["input"] + usage.output_tokens * HARGA_PER_JUTA["output"]) / 1e6


def teks(respon):
    return "".join(b.text for b in respon.content if b.type == "text")


def tool(nama, deskripsi, properti=None, wajib=None):
    """Buat definisi tool strict. Semua properti wajib kecuali `wajib` diisi."""
    properti = properti or {}
    return {"name": nama, "description": deskripsi, "strict": True,
            "input_schema": {"type": "object", "properties": properti,
                             "required": list(properti) if wajib is None else wajib, "additionalProperties": False}}


MAKS_BARIS = 50


def koneksi_baca_saja():
    return sqlite3.connect(f"{DB.resolve().as_uri()}?mode=ro", uri=True)


def lihat_skema():
    with koneksi_baca_saja() as con:
        return "\n\n".join(r[0] for r in con.execute("SELECT sql FROM sqlite_master WHERE type = 'table'"))


def jalankan_sql(query):
    con = koneksi_baca_saja()
    try:
        cur = con.execute(query)
        kolom = [d[0] for d in cur.description or []]
        baris = cur.fetchmany(MAKS_BARIS + 1)
    finally:
        con.close()
    return {"kolom": kolom, "baris": [list(b) for b in baris[:MAKS_BARIS]], "terpotong": len(baris) > MAKS_BARIS}


TOOL_SKEMA = tool("lihat_skema", "Tampilkan perintah CREATE TABLE semua tabel (kolom, tipe, dan komentar arti kolom). "
                                 "Panggil ini dulu jika belum tahu struktur database.")
TOOL_SQL = tool("jalankan_sql", "Jalankan SATU query SQLite SELECT (baca-saja) pada database toko dan kembalikan maksimal "
                                "50 baris. Gunakan agregasi (SUM, COUNT, AVG, GROUP BY) agar hasilnya ringkas.",
                {"query": {"type": "string", "description": "Satu query SQLite SELECT"}})


def jalankan_agen(pesan, tools, fungsi, system, max_langkah=10, max_biaya=0.50, tampilkan=True, **kwargs):
    """Loop agen generik (lihat notebook 01). `pesan`: tugas baru (str) atau riwayat messages (list).

    Mengembalikan (jawaban, jejak per langkah, messages).
    """
    messages = [{"role": "user", "content": pesan}] if isinstance(pesan, str) else list(pesan)
    jejak, total = [], 0.0
    for langkah in range(1, max_langkah + 1):
        r = client.beta.messages.create(model=MODEL, max_tokens=16000, system=system, tools=tools,
                                        messages=messages, **FALLBACK, **kwargs)
        messages.append({"role": "assistant", "content": r.content})
        total += biaya(r.usage)
        panggilan = [b for b in r.content if b.type == "tool_use"]
        jejak.append({"langkah": langkah, "token_input": r.usage.input_tokens, "token_output": r.usage.output_tokens,
                      "biaya": biaya(r.usage), "tool": [b.name for b in panggilan]})
        if tampilkan:
            for b in r.content:
                if b.type == "text" and b.text.strip():
                    print(f"💬 {b.text.strip()[:600]}")
        if r.stop_reason != "tool_use":
            awalan = "" if r.stop_reason == "end_turn" else f"[berhenti: {r.stop_reason}] "
            return awalan + teks(r), jejak, messages
        if total > max_biaya:                               # guardrail anggaran
            return f"[berhenti: anggaran US${max_biaya:.2f} habis]", jejak, messages
        hasil = []
        for b in panggilan:
            try:
                keluaran = fungsi[b.name](**b.input)
                hasil.append({"type": "tool_result", "tool_use_id": b.id,
                              "content": json.dumps(keluaran, ensure_ascii=False, default=str)})
            except Exception as e:  # error dikirim ke model agar bisa memperbaiki langkahnya
                hasil.append({"type": "tool_result", "tool_use_id": b.id, "content": f"Error: {e}", "is_error": True})
            if tampilkan:
                print(f"🔧 {b.name}({json.dumps(b.input, ensure_ascii=False)[:200]})\n   👀 {hasil[-1]['content'][:200]}")
        messages.append({"role": "user", "content": hasil})
    return "[berhenti: melebihi max_langkah]", jejak, messages


SYSTEM = """Kamu analis data Toko Elektronik Nusantara (TEN). Jawab pertanyaan bisnis dengan memeriksa database SQLite toko.
- Jika belum tahu strukturnya, lihat skema dulu. Pendapatan = SUM(jumlah * harga_satuan) dari pesanan berstatus 'selesai', kecuali diminta lain.
- Gunakan SQL untuk menghitung; jangan menebak angka.
- Jawaban akhir ringkas dalam bahasa Indonesia, sebutkan angka penting dalam rupiah dan asumsi yang kamu pakai."""
TOOLS_SQL = [TOOL_SKEMA, TOOL_SQL]
FUNGSI_SQL = {"lihat_skema": lihat_skema, "jalankan_sql": jalankan_sql}

## 1. Memori Jangka Pendek: Konteks yang Terus Membesar

Dalam satu sesi, "memori" agen adalah daftar `messages`. Setiap langkah menambah respons model dan hasil tool, lalu seluruhnya dikirim ulang. Kita lihat pertumbuhannya dalam satu percakapan beberapa giliran.

In [ ]:
import matplotlib.pyplot as plt

GILIRAN = ["Berapa pendapatan tahun 2025?", "Bagaimana dengan semester 1 2026?",
           "Kategori apa yang paling besar kontribusinya di semester 1 2026?"]
riwayat, token_per_langkah = [], []
for pertanyaan in GILIRAN[:2] if QUICK_RUN else GILIRAN:
    print(f"\n🧑 {pertanyaan}")
    riwayat = riwayat + [{"role": "user", "content": pertanyaan}]
    jawaban, jejak, riwayat = jalankan_agen(riwayat, TOOLS_SQL, FUNGSI_SQL, SYSTEM, tampilkan=False)
    token_per_langkah += [j["token_input"] for j in jejak]
    print(f"🤖 {jawaban[:400]}")

plt.figure(figsize=(7, 3))
plt.plot(range(1, len(token_per_langkah) + 1), token_per_langkah, "o-")
plt.xlabel("panggilan API ke-")
plt.ylabel("token input")
plt.title("Token input per panggilan dalam satu sesi")
plt.show()

Token input naik terus: biaya per panggilan makin mahal, dan suatu saat bisa mendekati batas context window. Strategi umum:
- **Batasi output tool** (seperti `MAKS_BARIS = 50`) dan minta agregasi di SQL, bukan data mentah.
- **Ringkas** (*compaction*): ganti riwayat panjang dengan ringkasan, lalu lanjutkan dari sana.
- Fitur API seperti **context editing** dan **compaction** di sisi server (beta) melakukan hal serupa secara otomatis untuk agen yang berjalan lama.

Satu hal yang perlu dihindari: **mengedit atau menghapus pesan lama** di tengah riwayat (misalnya memotong hasil tool lama). Ini merusak prompt cache, dan pada model terbaru blok *thinking* terikat pada riwayat yang persis sama. Jika riwayat perlu dipangkas, ganti **seluruhnya** dengan ringkasan, seperti di bawah ini.

In [ ]:
def ke_teks(messages, maks=400):
    """Ubah riwayat (termasuk blok tool) menjadi transkrip teks untuk diringkas."""
    baris = []
    for m in messages:
        isi = m["content"]
        if isinstance(isi, str):
            baris.append(f"{m['role']}: {isi}")
            continue
        for b in isi:
            tipe = b["type"] if isinstance(b, dict) else b.type
            if tipe == "text":
                baris.append(f"{m['role']}: {b.text}")
            elif tipe == "tool_use":
                baris.append(f"[memanggil {b.name} {json.dumps(b.input, ensure_ascii=False)}]")
            elif tipe == "tool_result":
                baris.append(f"[hasil tool: {str(b['content'])[:maks]}]")
    return "\n".join(baris)


def ringkas_riwayat(messages):
    r = client.beta.messages.create(
        model=MODEL, max_tokens=16000, output_config={"effort": "low"},
        system="Ringkas percakapan analisis data berikut. Pertahankan semua angka penting, definisi yang disepakati, "
               "dan pertanyaan yang belum selesai. Maksimal 150 kata.",
        messages=[{"role": "user", "content": f"<transkrip>\n{ke_teks(messages)}\n</transkrip>"}], **FALLBACK)
    return teks(r)


ringkasan = ringkas_riwayat(riwayat)
print(ringkasan, "\n")

lanjutan = "Berapa persen pertumbuhan semester 1 2026 dibanding semester 1 2025?"
hitung_token = client.beta.messages.count_tokens
penuh = hitung_token(model=MODEL, system=SYSTEM, tools=TOOLS_SQL,
                     messages=riwayat + [{"role": "user", "content": lanjutan}]).input_tokens
ringkas = hitung_token(model=MODEL, system=SYSTEM, tools=TOOLS_SQL, messages=[
    {"role": "user", "content": f"<ringkasan_sebelumnya>\n{ringkasan}\n</ringkasan_sebelumnya>\n\n{lanjutan}"}]).input_tokens
print(f"Token input untuk pertanyaan berikutnya: riwayat penuh {penuh:,} vs dari ringkasan {ringkas:,}")

Ringkasan menghemat token, tetapi **ada informasi yang hilang** (detail query, angka antara). Karena itu compaction biasanya baru dilakukan saat konteks sudah panjang, bukan di setiap giliran.

## 2. Memori Jangka Panjang: Catatan Antarsesi

Riwayat hilang begitu sesi berakhir. Untuk mengingat hal-hal yang stabil (preferensi pengguna, definisi bisnis, temuan penting), beri agen tool untuk **menyimpan** dan **membaca** catatan di penyimpanan permanen. Di sini cukup file JSON; di produksi bisa database atau vector DB (modul 07).

Claude API juga menyediakan *memory tool* standar yang bekerja dengan prinsip yang sama: agen membaca dan menulis file di direktori memori, dan aplikasimu yang menentukan penyimpanannya.

In [ ]:
FILE_INGATAN = Path("ingatan.json")
FILE_INGATAN.unlink(missing_ok=True)          # mulai dari memori kosong untuk demo


def baca_ingatan():
    return json.loads(FILE_INGATAN.read_text()) if FILE_INGATAN.exists() else {}


def simpan_ingatan(topik, isi):
    ingatan = baca_ingatan()
    ingatan[topik] = isi
    FILE_INGATAN.write_text(json.dumps(ingatan, ensure_ascii=False, indent=2))
    return f"tersimpan: {topik}"


TOOLS_MEMORI = TOOLS_SQL + [
    tool("baca_ingatan", "Baca semua catatan jangka panjang tentang pengguna dan preferensinya."),
    tool("simpan_ingatan", "Simpan catatan jangka panjang yang stabil (preferensi pengguna, definisi yang disepakati). "
                           "Jangan menyimpan data pribadi pelanggan atau hasil query sementara.",
         {"topik": {"type": "string", "description": "Judul singkat, misalnya 'format_angka'"},
          "isi": {"type": "string"}}),
]
FUNGSI_MEMORI = {**FUNGSI_SQL, "baca_ingatan": baca_ingatan, "simpan_ingatan": simpan_ingatan}
SYSTEM_MEMORI = SYSTEM + """
- Di awal setiap sesi, panggil baca_ingatan dan ikuti preferensi yang tersimpan.
- Jika pengguna menyatakan preferensi atau kesepakatan yang berlaku seterusnya, simpan dengan simpan_ingatan."""

print("=== Sesi 1 ===")
jawaban, _, _ = jalankan_agen("Mulai sekarang, tampilkan angka pendapatan dalam juta rupiah (dibulatkan) dan "
                              "fokuskan analisis pada kota Surabaya. Tolong diingat untuk sesi berikutnya.",
                              TOOLS_MEMORI, FUNGSI_MEMORI, SYSTEM_MEMORI)
print(f"🤖 {jawaban}\n\nIsi ingatan.json: {baca_ingatan()}")

In [ ]:
print("=== Sesi 2 (riwayat baru, kosong) ===")
jawaban, _, _ = jalankan_agen("Berapa pendapatan bulan Mei 2026?", TOOLS_MEMORI, FUNGSI_MEMORI, SYSTEM_MEMORI)
print(f"🤖 {jawaban}")

Di sesi 2 agen tidak menerima riwayat sesi 1 sama sekali, tetapi seharusnya tetap menjawab dalam juta rupiah dan fokus pada Surabaya karena membaca `ingatan.json`.

Memori permanen juga membawa risiko: apa pun yang tersimpan akan memengaruhi **semua** sesi berikutnya. Beri pengguna cara melihat dan menghapus memorinya, jangan simpan data sensitif, dan waspadai teks jahat yang "menyusup" ke memori (prompt injection yang bertahan lama).

## 3. State Tugas: Rencana yang Bisa Diperbarui

Untuk tugas panjang, agen bekerja lebih terarah jika menulis **rencana** lebih dulu lalu mencentang langkah yang selesai. State rencana disimpan di kode kita (bukan hanya di teks percakapan), sehingga bisa ditampilkan ke pengguna, disimpan, atau dilanjutkan jika proses terputus.

In [ ]:
RENCANA = []


def tulis_rencana(langkah):
    RENCANA.clear()
    RENCANA.extend({"no": i, "langkah": isi, "status": "belum", "catatan": ""} for i, isi in enumerate(langkah, 1))
    return RENCANA


def perbarui_langkah(nomor, status, catatan):
    for item in RENCANA:
        if item["no"] == nomor:
            item.update(status=status, catatan=catatan)
            return item
    raise ValueError(f"langkah {nomor} tidak ada; rencana punya {len(RENCANA)} langkah")


TOOLS_RENCANA = TOOLS_SQL + [
    tool("tulis_rencana", "Tulis rencana kerja (3-6 langkah) sebelum mengerjakan tugas yang panjang.",
         {"langkah": {"type": "array", "items": {"type": "string"}}}),
    tool("perbarui_langkah", "Tandai status satu langkah rencana setelah dikerjakan.",
         {"nomor": {"type": "integer"}, "status": {"type": "string", "enum": ["selesai", "dilewati"]},
          "catatan": {"type": "string", "description": "Temuan singkat dari langkah ini"}}),
]
FUNGSI_RENCANA = {**FUNGSI_SQL, "tulis_rencana": tulis_rencana, "perbarui_langkah": perbarui_langkah}

tugas = ("Buat laporan singkat kinerja semester 1 2026: total pendapatan, 3 produk dengan pendapatan tertinggi, "
         "kota dengan pertumbuhan pendapatan tertinggi dibanding semester 1 2025, dan persentase pesanan yang dikembalikan. "
         "Tulis rencana dulu, lalu perbarui status setiap langkah.")
laporan, jejak, _ = jalankan_agen(tugas, TOOLS_RENCANA, FUNGSI_RENCANA, SYSTEM, max_langkah=6 if QUICK_RUN else 20,
                                  tampilkan=False)
display(pd.DataFrame(RENCANA) if RENCANA else "Agen tidak menulis rencana.")
print(f"🤖 {laporan}\n\n{len(jejak)} panggilan API, biaya ≈ US${sum(j['biaya'] for j in jejak):.3f}")

## 4. Model Context Protocol (MCP)

Sejauh ini tool kita adalah fungsi Python di notebook ini. Jika ingin memakai tool yang sama di Claude Desktop, Claude Code, IDE, atau aplikasi lain, kita harus menulis ulang integrasinya di setiap tempat. **MCP** adalah protokol terbuka yang menstandarkan cara aplikasi AI (*client*) menemukan dan memanggil tool, data (*resource*), dan template prompt dari sebuah **server**.

```
Claude Desktop ─┐                       ┌─ server_toko.py  (database toko)
Claude Code  ───┼──── protokol MCP ─────┼─ server GitHub, Slack, filesystem, ...
Notebook ini ───┘   (stdio atau HTTP)   └─ server buatanmu sendiri
```

Tulis server sekali, pakai di semua client. File [`server_toko.py`](server_toko.py) adalah server MCP untuk database toko, dibuat dengan SDK resmi `mcp`. Tool didefinisikan dengan dekorator `@server.tool()`; nama, deskripsi (docstring), dan schema argumen (type hint) dibuat otomatis:

In [ ]:
print(Path("server_toko.py").read_text())

Notebook ini berperan sebagai **client**: menjalankan server sebagai proses terpisah (transport **stdio**), lalu menanyakan daftar tool-nya. Kode MCP bersifat *async*; Jupyter/Colab mendukung `await` dan `async with` langsung di sel.

In [ ]:
import sys

from anthropic.lib.tools.mcp import async_mcp_tool
from mcp import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client

params = StdioServerParameters(command=sys.executable, args=["server_toko.py"], env={**os.environ, "TOKO_DB": str(DB)})

async with stdio_client(params) as (baca, tulis):
    async with ClientSession(baca, tulis) as sesi:
        await sesi.initialize()
        tools_mcp = (await sesi.list_tools()).tools
        for t in tools_mcp:
            print(f"🔌 {t.name}: {t.description.strip().splitlines()[0]}")
        hasil = await sesi.call_tool("jalankan_sql", {"query": "SELECT kategori, COUNT(*) FROM produk GROUP BY kategori"})
        print("\nPanggilan langsung:", hasil.content[0].text)

Sekarang hubungkan tool MCP ke Claude. Helper `async_mcp_tool` dari SDK `anthropic` mengubah tool MCP menjadi tool untuk **tool runner** (modul 08), sehingga kita tidak perlu menulis loop sendiri. Setiap pemanggilan tool diteruskan ke server MCP.

In [ ]:
aclient = anthropic.AsyncAnthropic(api_key=api_key)

async with stdio_client(params) as (baca, tulis):
    async with ClientSession(baca, tulis) as sesi:
        await sesi.initialize()
        tools_mcp = (await sesi.list_tools()).tools
        runner = aclient.beta.messages.tool_runner(
            model=MODEL, max_tokens=16000, system=SYSTEM,
            tools=[async_mcp_tool(t, sesi) for t in tools_mcp],
            messages=[{"role": "user", "content": "Produk apa yang paling sering diulas dengan rating 1?"}],
            **FALLBACK)
        async for pesan in runner:
            for b in pesan.content:
                if b.type == "tool_use":
                    print(f"🔧 (lewat MCP) {b.name}({json.dumps(b.input, ensure_ascii=False)[:200]})")
                elif b.type == "text" and b.text.strip():
                    print(f"🤖 {b.text.strip()}")

Server yang sama bisa dipasang di **Claude Desktop** atau **Claude Code** dengan menambahkan konfigurasi server MCP (perintah `python server_toko.py` beserta `TOKO_DB`). Setelah itu kamu bisa bertanya tentang data toko langsung dari aplikasi tersebut. Selain server lokal (stdio), MCP juga mendukung server jarak jauh lewat HTTP, dan Claude API punya fitur *MCP connector* untuk terhubung langsung ke server MCP jarak jauh.

Ingat bahwa server MCP menjalankan kode di komputermu dengan izin prosesnya. Pasang hanya server dari sumber tepercaya, dan buat server sendiri dengan prinsip **least privilege** (server kita hanya bisa membaca).

## 5. Framework Agent

Kita membangun semuanya dari nol agar paham mekanismenya. Dalam praktik, pilihannya:

| Pilihan | Cocok untuk |
|---|---|
| Loop sendiri (notebook 01) | Belajar, kontrol penuh, kebutuhan khusus |
| **Tool runner** SDK `anthropic` | Loop tool standar dengan kode minimal (modul 08, bagian MCP di atas) |
| **Claude Agent SDK** | Agen dengan kemampuan seperti Claude Code (file, terminal, MCP, subagent) di aplikasimu sendiri |
| Framework umum (LangGraph, dll.) | Graf alur kerja kompleks, banyak agen, integrasi ekosistem tertentu |
| **Managed Agents** (Anthropic) | Loop dan sandbox eksekusi dijalankan di infrastruktur Anthropic |

Framework mempercepat pekerjaan, tetapi konsep dasarnya tetap sama: loop, tool, riwayat/memori, dan kondisi berhenti. Jika sesuatu tidak berjalan sesuai harapan, pemahaman dari notebook 01 adalah kunci untuk men-*debug*-nya.

## Ringkasan

- **Memori jangka pendek** = riwayat `messages`; tumbuh setiap langkah. Kelola dengan membatasi output tool dan meringkas seluruh riwayat bila perlu, bukan mengedit pesan lama.
- **Memori jangka panjang** = tool baca/tulis ke penyimpanan permanen; berguna untuk preferensi, tetapi harus bisa dilihat/dihapus pengguna dan dijaga dari data sensitif.
- **State tugas** (rencana) disimpan di kode agar bisa dipantau dan dilanjutkan.
- **MCP** menstandarkan tool: tulis server sekali, pakai dari banyak client; `async_mcp_tool` menghubungkannya ke tool runner Claude.

## Latihan

1. **Compaction otomatis.** Ubah `jalankan_agen` agar ketika `token_input` melebihi ambang (misalnya 8.000 token), riwayat diganti ringkasan dari `ringkas_riwayat` lalu loop dilanjutkan. Uji dengan percakapan 6 giliran dan bandingkan total biaya dengan versi tanpa compaction. Apakah kualitas jawaban berubah?
2. **Memori yang bisa dikelola pengguna.** Tambahkan tool `hapus_ingatan(topik)` dan perintah khusus: jika pengguna mengetik "apa yang kamu ingat tentang saya?", agen menampilkan seluruh isi memori. Lalu coba simpan preferensi yang bertentangan dengan preferensi lama. Bagaimana agen menanganinya?
3. **Server MCP kamu sendiri.** Tambahkan tool `pendapatan_bulanan(tahun: int)` ke `server_toko.py` yang mengembalikan pendapatan per bulan. Tambahkan juga sebuah *resource* (lihat dokumentasi SDK `mcp`) berisi penjelasan definisi bisnis (misalnya definisi "pendapatan"). Hubungkan server ke Claude Desktop atau Claude Code, lalu ajukan pertanyaan dari sana.